In [ ]:
seq = np.arange(15)
import keras

sampling_rate = 1
sequence_length = 3
delay = 3
batch_size = 2

dataset = keras.utils.timeseries_dataset_from_array(
    data=seq[:-delay],
    targets=seq[delay:],
    sequence_length=sequence_length,
    sampling_rate=sampling_rate,
    batch_size=batch_size,
    start_index=0,
    end_index=len(seq[:-delay])-1
)

for x,t in dataset:
    for i in range(x.shape[0]):
        print([int(a) for a in x[i]], int(t[i]))


In [ ]:

for x,t in dataset:
    print("X:", x.numpy().tolist(),"    y:",t.numpy().tolist())        

In [ ]:
#######################

In [ ]:
import os
fname = os.path.join("d:jena_climate_2009_2016.csv")

In [ ]:
with open(fname) as f:
    data = f.read()

In [ ]:
lines = data.split("\n")
lines[0]

In [ ]:
lines[1:9]

In [ ]:
header = lines[0].split(",")
header

In [ ]:
len(header)

In [ ]:
lines = lines[1:]

In [ ]:
len(lines)

In [ ]:
import numpy as np
temperature = np.zeros(len(lines))

In [ ]:
raw_data = np.zeros((len(lines), len(header)-1))
raw_data

In [ ]:
for i, line in enumerate(lines):
    values = [float(x) for x in line.split(",")[1:]]
    temperature[i] = values[1]
    raw_data[i,:] = values[:]    

In [ ]:
temperature

In [ ]:
import matplotlib.pyplot as plt 
plt.plot(range(len(temperature)), temperature)
plt.show()

In [ ]:
6 * 24 * 10

In [ ]:
plt.plot(range(1440), temperature[:1440])  # 6 * 24 * 10
plt.show()

In [ ]:
len(raw_data)

In [ ]:
num_train_samples = int(0.5 * len(raw_data))
num_train_samples

In [ ]:
num_val_samples = int(0.25 * len(raw_data))
num_val_samples

In [ ]:
num_test_samples = len(raw_data) - num_train_samples - num_val_samples
num_test_samples

In [ ]:
mean = raw_data[:num_train_samples].mean(axis=0)
raw_data -= mean

std = raw_data[:num_train_samples].std(axis=0)
raw_data /= std

In [ ]:

sampling_rate = 6
sequence_length = 120  # 5 * 24 = 120     : 5 days
delay = sampling_rate * (sequence_length + 24 -1)
delay

In [ ]:
import keras 

sampling_rate = 6
sequence_length = 120                                  # 5 * 24 = 120     : 5 days
delay = sampling_rate * (sequence_length + 24 -1)
batch_size = 256

In [ ]:

train_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=raw_data[delay:],
    sequence_length=sequence_length,
    sampling_rate=sampling_rate,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples,
    shuffle=True
)


In [ ]:

val_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=raw_data[delay:],
    sequence_length=sequence_length,
    sampling_rate=sampling_rate,
    batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples+ num_val_samples,
    shuffle=True
)

In [ ]:

test_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=raw_data[delay:],
    sequence_length=sequence_length,
    sampling_rate=sampling_rate,
    batch_size=batch_size,
    start_index=num_train_samples + num_val_samples,
    shuffle=True
)

In [ ]:
for x,t in train_dataset:
    print(x.shape)
    print(t.shape)
    break

In [ ]:
raw_data.shape

In [ ]:
from keras import layers

inputs = keras.Input(shape=(120, 14))
x = layers.LSTM(16)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)

model = keras.Model(inputs, outputs)

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
callbacks = [keras.callbacks.ModelCheckpoint("a.keras", save_best_only=True)]

In [ ]:
h = model.fit(train_dataset, epochs=10, validation_data=val_dataset, callbacks=callbacks)

In [ ]:
best_model = keras.model.load_model('a.keras')
best_model.evaluate(test_dataset)

In [ ]:
len(loss)

In [ ]:
import matplotlib.pyplot as plt 

loss = h.history['mae']
val_loss = h.history['val_mae']

plt.plot(range(1,11), loss, 'r')
plt.plot(range(1,11), val_loss ,'b')

plt.show()

In [ ]:
##############

In [ ]:
##############

In [ ]:
##############

In [ ]:
# mlp

inputs = keras.Input(shape=(120, 14))
x = layers.Flatten()(inputs)
x = layers.Dense(16, activation='relu')(x)
outputs = layers.Dense(1)(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
h = model.fit(train_dataset, epochs=10, validation_data=val_dataset)

model.evaluate(test_dataset)

In [ ]:
# CNN
inputs = keras.Input(shape=(120, 14))

x = layers.Conv1D(8, 24 , activation='relu')(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 12, activation='relu')(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 6, activation='relu')(inputs)
x = layers.GlobalAveragePooling1D()(x)

outputs = layers.Dense(1)(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
h = model.fit(train_dataset, epochs=10, validation_data=val_dataset)

model.evaluate(test_dataset)